# PyTorch Exercises — Layers, Models, Loss Functions & Training

*ML & NLP course — Data Trainers LLC — Axel Sirota*

## Welcome to PyTorch

PyTorch is the framework behind GPT-4, LLaMA, Stable Diffusion, and virtually every cutting-edge model you will encounter in this course. Unlike Keras, where the framework does a lot of magic for you, PyTorch puts you in control — you write the training loop, you manage gradients, you decide what happens at every step.

That control is exactly what makes PyTorch the professional standard. Once you know these primitives, you can read (and write) any research code.

## What you will practice today

1. **Creating and configuring layers** — `nn.Linear`, `nn.ReLU`, `nn.Dropout`, `nn.Conv2d`
2. **Building sequential models** — `nn.Sequential` for straight pipelines
3. **Custom `nn.Module` subclassing** — the PyTorch equivalent of Keras' Functional API, enabling multi-branch architectures
4. **Custom loss functions** — MSE from scratch, regularized loss, both as `nn.Module` and plain functions
5. **The training loop** — the canonical 6-line loop (`zero_grad → forward → loss → backward → step`)
6. **Freezing layers** — `requires_grad = False` for transfer learning

## Prerequisites

- Basic Python / NumPy
- Familiarity with the concept of a neural network (what a layer is, what a loss is)
- No prior PyTorch required — we start from zero

## How to run

- **Google Colab (recommended):** Runtime → Change runtime type → GPU (T4 is free and more than enough for these exercises)
- **Local virtualenv:** Activate your `.venv` and `pip install torch torchvision`

Runtime: ~30–45 minutes for all exercises.

## Section 0 — Environment Setup

Run the cell below first. It installs packages (Colab-safe — skips if already present), imports everything we need, pins random seeds for reproducibility, and detects whether a GPU is available.

In [ ]:
# Install required packages (safe to run on Colab; skips silently if already installed)
!pip install -q torch torchvision numpy matplotlib

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

# ---- reproducibility ----
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ---- device detection (use GPU when available) ----
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch version : {torch.__version__}")
print(f"Using device    : {device}")
if device.type == 'cuda':
    print(f"GPU name        : {torch.cuda.get_device_name(0)}")

print("\nEnvironment setup complete!")

## What Are We Building Today?

Imagine you are a new ML engineer at a computer-vision startup. Your first ticket reads:

> "We need to classify handwritten digits (MNIST) and also fit a regression model on a noisy sine wave. Use PyTorch. The lead engineer will review your code."

By the end of this notebook you will have built both models from scratch — the digit classifier and the sine-wave regressor — using every primitive that ticket implies: layers, sequential stacks, custom modules, custom losses, a training loop, and transfer-learning freeze/unfreeze patterns.

None of the individual pieces are hard. The goal is to build *muscle memory* for the PyTorch API so that when you hit the real course notebooks (CBOW, BERT, LSTM…) the framework never slows you down.

In [ ]:
# Quick dataset preview — MNIST (28x28 grayscale digits, 10 classes)
from torchvision import datasets, transforms

# Download MNIST (cached after first run)
mnist_train = datasets.MNIST(root='/tmp/mnist', train=True,
                              download=True,
                              transform=transforms.ToTensor())
mnist_test  = datasets.MNIST(root='/tmp/mnist', train=False,
                              download=True,
                              transform=transforms.ToTensor())

print(f"Training samples : {len(mnist_train):,}")
print(f"Test samples     : {len(mnist_test):,}")
print(f"Image shape      : {mnist_train[0][0].shape}  (C, H, W)")
print(f"Label example    : {mnist_train[0][1]}")

# Show a grid of 8 random digits
fig, axes = plt.subplots(1, 8, figsize=(12, 2))
indices = np.random.choice(len(mnist_train), 8, replace=False)
for ax, idx in zip(axes, indices):
    img, label = mnist_train[idx]
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title(str(label), fontsize=10)
    ax.axis('off')
plt.suptitle("MNIST — sample digits", y=1.02)
plt.tight_layout()
plt.show()

## Section 1 — Creating and Configuring Layers

### PyTorch's building blocks: `torch.nn`

Every neural network layer in PyTorch lives in `torch.nn`. The three you will use most often are:

| Layer | What it does | Key arguments |
|---|---|---|
| `nn.Linear(in, out)` | Fully-connected: `y = xW^T + b` | `in_features`, `out_features`, `bias=True` |
| `nn.ReLU()` | Element-wise `max(0, x)` | — |
| `nn.Dropout(p)` | Randomly zeroes activations during training | `p` = drop probability |
| `nn.Conv2d(in_ch, out_ch, k)` | 2-D convolution | `in_channels`, `out_channels`, `kernel_size` |

All layers are **objects** — you create them once and call them like functions:

```python
layer = nn.Linear(784, 256)   # create
y = layer(x)                   # call (runs the forward pass)
```

Layers track their own **parameters** (weights and biases). You never touch those arrays manually — PyTorch's autograd system updates them through `optimizer.step()`.

### Demo — instantiate and inspect each layer

In [ ]:
# Demo: create layers and inspect their shapes

# 1. A fully-connected layer: maps 784 inputs (flattened 28x28 image) to 256 outputs
fc = nn.Linear(in_features=784, out_features=256)
print("nn.Linear(784, 256)")
print(f"  weight shape : {fc.weight.shape}   # (out_features, in_features)")
print(f"  bias shape   : {fc.bias.shape}")

# 2. ReLU — no parameters, just an operation
relu = nn.ReLU()
demo_tensor = torch.tensor([-2.0, -1.0, 0.0, 1.0, 2.0])
print(f"\nnn.ReLU() on {demo_tensor.tolist()} -> {relu(demo_tensor).tolist()}")

# 3. Dropout — stochastically zeroes p fraction of activations (only active in .train() mode)
drop = nn.Dropout(p=0.2)
demo_ones = torch.ones(10)
print(f"\nnn.Dropout(0.2) on all-ones (train mode): {drop(demo_ones).tolist()}")
# In eval mode, Dropout is a no-op
drop.eval()
print(f"nn.Dropout(0.2) on all-ones (eval mode) : {drop(demo_ones).tolist()}")

# 4. Conv2d: 32 filters, 3x3 kernel, 1-channel (grayscale) input
conv = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1)
print(f"\nnn.Conv2d(1, 32, 3, padding=1)")
print(f"  weight shape : {conv.weight.shape}   # (out_ch, in_ch, kH, kW)")
print(f"  bias shape   : {conv.bias.shape}")

# Run a fake MNIST batch (4 images, 1 channel, 28x28) through the conv
fake_batch = torch.randn(4, 1, 28, 28)
out = conv(fake_batch)
print(f"  input  shape : {fake_batch.shape}")
print(f"  output shape : {out.shape}   # (batch, 32, 28, 28) — padding preserves spatial size")

### Lab 1 — Create and configure layers

Complete each `None` below. After filling them in, run the cell — the verification prints will confirm each layer has the expected parameter shape.

**Tasks:**
1. Create `fc_layer`: a `Linear` layer mapping **784 → 10** (MNIST: 784 pixel inputs, 10 digit classes).
2. Create `relu_layer`: a `ReLU` activation (no arguments needed).
3. Create `dropout_layer`: a `Dropout` layer with rate **0.3**.
4. Create `conv_layer`: a `Conv2d` with **1 input channel**, **32 output channels**, **3×3 kernel**, and `padding=1`.

In [ ]:
# Lab 1: creating and configuring layers

# 1. Fully-connected layer: 784 inputs -> 10 outputs
fc_layer = None  # YOUR CODE

# 2. ReLU activation
relu_layer = None  # YOUR CODE

# 3. Dropout with rate 0.3
dropout_layer = None  # YOUR CODE

# 4. Conv2d: 1 input channel, 32 output channels, 3x3 kernel, padding=1
conv_layer = None  # YOUR CODE

# --- Verification (do not modify below this line) ---
if fc_layer is not None:
    print(f"fc_layer weight shape    : {fc_layer.weight.shape}   (expected torch.Size([10, 784]))")
if relu_layer is not None:
    test = torch.tensor([-1.0, 0.0, 1.0])
    print(f"relu_layer(-1,0,1)       : {relu_layer(test).tolist()}   (expected [0.0, 0.0, 1.0])")
if dropout_layer is not None:
    print(f"dropout_layer.p          : {dropout_layer.p}   (expected 0.3)")
if conv_layer is not None:
    print(f"conv_layer weight shape  : {conv_layer.weight.shape}   (expected torch.Size([32, 1, 3, 3]))")
    probe = torch.randn(1, 1, 28, 28)
    print(f"conv_layer output shape  : {conv_layer(probe).shape}   (expected torch.Size([1, 32, 28, 28]))")

## Section 2 — Building Sequential Models

### `nn.Sequential`: stacking layers in a pipeline

When your network is a straight chain — input → layer → activation → dropout → ... → output — `nn.Sequential` is the cleanest way to express it. You pass the layers in order and PyTorch wires them up automatically:

```python
model = nn.Sequential(
    nn.Linear(784, 256),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(256, 10),
)
```

That is the entire digit classifier: one forward call runs the tensor through all four layers in order.

### Shape arithmetic

A critical habit: always trace the tensor shape through your architecture before writing any code.

For our MNIST classifier:
- Input: `(batch, 1, 28, 28)` — a batch of grayscale images
- After `Flatten`: `(batch, 784)` — 28×28 = 784 pixels
- After `Linear(784, 256)` + `ReLU`: `(batch, 256)`
- After `Dropout(0.2)`: `(batch, 256)` — shape unchanged, some values zeroed
- After `Linear(256, 10)`: `(batch, 10)` — 10 raw logits, one per digit class

We **do not** apply softmax here. `nn.CrossEntropyLoss` expects raw logits and applies log-softmax internally. Applying softmax yourself and then using `CrossEntropyLoss` is a very common bug that silently destroys training — avoid it.

### Demo — a two-layer MLP

In [ ]:
# Demo: build a sequential MLP and trace shapes

# nn.Flatten converts (B, 1, 28, 28) -> (B, 784) automatically
demo_mlp = nn.Sequential(
    nn.Flatten(),           # (B, 1, 28, 28) -> (B, 784)
    nn.Linear(784, 128),    # 784 -> 128
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(128, 10),     # 128 -> 10 logits (no softmax!)
)

# Run a fake batch of 8 images through the model
fake_imgs = torch.randn(8, 1, 28, 28)
logits = demo_mlp(fake_imgs)

print(f"Input  shape : {fake_imgs.shape}")
print(f"Output shape : {logits.shape}   # (8 images, 10 class logits)")
print(f"\nModel architecture:")
print(demo_mlp)

# Count total trainable parameters
n_params = sum(p.numel() for p in demo_mlp.parameters() if p.requires_grad)
print(f"\nTotal trainable parameters: {n_params:,}")

### Lab 2 — Build a sequential MNIST classifier

Build `mnist_sequential` as an `nn.Sequential` with this exact architecture:

1. `Flatten` — reshapes `(B, 1, 28, 28)` to `(B, 784)`
2. `Linear(784, 256)` + `ReLU()`
3. `Dropout(0.2)`
4. `Linear(256, 128)` + `ReLU()`
5. `Dropout(0.2)`
6. `Linear(128, 10)` — raw logits, **no softmax**

After building it, the verification block checks the output shape and parameter count.

In [ ]:
# Lab 2: build a sequential MNIST classifier

mnist_sequential = None  # YOUR CODE — nn.Sequential(...)

# --- Verification ---
if mnist_sequential is not None:
    probe = torch.randn(4, 1, 28, 28)
    out = mnist_sequential(probe)
    print(f"Output shape : {out.shape}   (expected torch.Size([4, 10]))")
    n = sum(p.numel() for p in mnist_sequential.parameters() if p.requires_grad)
    print(f"Parameters   : {n:,}   (expected 235,146)")
    print("Architecture:")
    print(mnist_sequential)

## Section 3 — Custom `nn.Module` Subclassing

### When `nn.Sequential` is not enough

`nn.Sequential` works beautifully for linear pipelines, but real architectures often need **branches**: two parallel paths that process input differently and then merge. Examples:

- **ResNets**: a skip connection adds the input directly to the output of a few layers
- **Multi-modal models**: one branch encodes an image, another encodes text, then they concatenate
- **Siamese networks**: two identical encoders share weights and compare their outputs

For all of these you subclass `nn.Module` directly. The pattern is always the same:

```python
class MyModel(nn.Module):
    def __init__(self):
        super().__init__()        # mandatory — registers parameters
        self.layer1 = nn.Linear(...)
        self.layer2 = nn.Linear(...)

    def forward(self, x):
        # define the computation graph here
        out = self.layer1(x)
        out = self.layer2(out)
        return out
```

Two rules:
1. Always call `super().__init__()` — without it, parameters won't be registered and `optimizer` won't see them.
2. Put all layers that have learnable parameters as `self.xxx` attributes — this is how PyTorch discovers them.

### Demo — a dual-branch MNIST model

This model runs the input through two parallel linear branches, concatenates their outputs, then passes the combined vector through a final classifier layer.

In [ ]:
# Demo: dual-branch module (illustrates why nn.Sequential is insufficient)

class DualBranchMNIST(nn.Module):
    """Two parallel FC branches that process the flattened image independently.
    Their outputs are concatenated and fed to a shared classifier head.

    Architecture:
        input (784) ─┬─ branch_a: Linear(784,128) + ReLU ─┐
                     └─ branch_b: Linear(784,64)  + ReLU ─┤ cat -> Linear(192,10)
    """

    def __init__(self):
        super().__init__()  # CRITICAL: registers all child modules as parameters

        # Branch A: wider representation
        self.branch_a = nn.Sequential(
            nn.Linear(784, 128),
            nn.ReLU(),
        )
        # Branch B: narrower representation — a different feature view of the same input
        self.branch_b = nn.Sequential(
            nn.Linear(784, 64),
            nn.ReLU(),
        )
        # Classifier head: takes concatenated branches (128 + 64 = 192) -> 10 classes
        self.classifier = nn.Linear(192, 10)

    def forward(self, x):
        # x shape: (B, 1, 28, 28) — flatten to (B, 784) first
        x = x.view(x.size(0), -1)          # (B, 784)
        a = self.branch_a(x)               # (B, 128)
        b = self.branch_b(x)               # (B, 64)
        combined = torch.cat([a, b], dim=1) # (B, 192)  — concatenate along feature dim
        return self.classifier(combined)    # (B, 10)   — raw logits

# Instantiate and run a shape check
demo_branch = DualBranchMNIST()
probe = torch.randn(8, 1, 28, 28)
out = demo_branch(probe)
print(f"Input  : {probe.shape}")
print(f"Output : {out.shape}   (expected: (8, 10))")
n = sum(p.numel() for p in demo_branch.parameters())
print(f"Params : {n:,}")

### Lab 3 — Implement a residual (skip-connection) block

Residual connections are the core of ResNets and transformers. The idea: add the **input** directly to the output of a transformation:

```
output = F(x) + x        # "skip connection" or "residual"
```

This means gradients can flow directly back through the addition operation, allowing very deep networks to train without vanishing gradients.

Your task: implement `ResidualBlock` below. It takes a vector of `dim` features, passes it through two linear layers with ReLU, and adds the original input before returning.

Architecture:
```
x (dim,) -> Linear(dim, dim) -> ReLU -> Linear(dim, dim) -> + x -> ReLU -> output
```

In [ ]:
# Lab 3: implement a residual block

class ResidualBlock(nn.Module):
    """A single residual block: two linear layers with a skip connection.

    forward(x) should compute:  ReLU(Linear2(ReLU(Linear1(x))) + x)
    """

    def __init__(self, dim):
        super().__init__()
        # 1. First linear layer: dim -> dim
        self.fc1 = None  # YOUR CODE
        # 2. Second linear layer: dim -> dim
        self.fc2 = None  # YOUR CODE

    def forward(self, x):
        # 3. Pass x through fc1 then ReLU
        out = None  # YOUR CODE
        # 4. Pass through fc2 (no activation yet)
        out = None  # YOUR CODE
        # 5. Add the skip connection (original x) and apply final ReLU
        out = None  # YOUR CODE
        return out


# --- Verification ---
if ResidualBlock.__init__ is not object.__init__:
    try:
        block = ResidualBlock(dim=64)
        probe = torch.randn(4, 64)    # batch of 4, feature dim 64
        result = block(probe)
        print(f"Input  shape : {probe.shape}")
        print(f"Output shape : {result.shape}   (expected torch.Size([4, 64]))")
        # Output should be >= 0 everywhere (final ReLU)
        print(f"Min value    : {result.min().item():.4f}   (expected >= 0.0)")
        n = sum(p.numel() for p in block.parameters())
        print(f"Parameters   : {n:,}   (expected {2 * 64*64 + 2*64:,})")
    except Exception as e:
        print(f"Error: {e}")

## Section 4 — Custom Loss Functions

### Why custom losses?

PyTorch ships with the losses you need 90% of the time (`CrossEntropyLoss`, `MSELoss`, `BCELoss`…). But sometimes you need to extend them:

- **Regularized loss**: add L1 or L2 weight penalties directly into the training objective
- **Domain-specific loss**: focal loss for imbalanced classes, contrastive loss for embeddings, ELBO for VAEs
- **Research**: you're implementing a new paper

There are two styles:

**Style 1 — plain function** (simplest, fine for stateless losses):
```python
def my_mse(predictions, targets):
    return ((predictions - targets) ** 2).mean()
```

**Style 2 — `nn.Module` subclass** (preferred when your loss has parameters or state):
```python
class RegularizedMSE(nn.Module):
    def __init__(self, lambda_l2):
        super().__init__()
        self.lambda_l2 = lambda_l2

    def forward(self, preds, targets, model):
        mse = ((preds - targets) ** 2).mean()
        l2  = sum(p.pow(2).sum() for p in model.parameters())
        return mse + self.lambda_l2 * l2
```

We will use a **sine-wave regression** problem to keep things concrete and fast: given x values, predict `sin(x) + noise`.

### Demo — MSE from scratch vs. `nn.MSELoss`

In [ ]:
# Demo: custom MSE vs built-in, and a toy sine regression dataset

# ---- Sine-wave regression dataset ----
N = 1000                              # number of data points
x_np = np.linspace(-2 * np.pi, 2 * np.pi, N).astype(np.float32)
y_np = np.sin(x_np) + 0.1 * np.random.randn(N).astype(np.float32)

x_tensor = torch.from_numpy(x_np).unsqueeze(1)   # (N, 1)
y_tensor = torch.from_numpy(y_np).unsqueeze(1)   # (N, 1)

plt.figure(figsize=(9, 3))
plt.scatter(x_np[::5], y_np[::5], s=5, alpha=0.5, label='noisy data')
plt.plot(x_np, np.sin(x_np), 'r-', lw=2, label='true sin(x)')
plt.legend(); plt.title('Sine regression dataset'); plt.tight_layout(); plt.show()

# ---- MSE comparison ----
def mse_scratch(preds, targets):
    """Mean squared error computed manually."""
    return ((preds - targets) ** 2).mean()

torch_mse = nn.MSELoss()

preds  = torch.tensor([0.5, 1.0, -0.5])
targets = torch.tensor([0.0, 1.0,  0.0])

custom_val = mse_scratch(preds, targets)
builtin_val = torch_mse(preds, targets)
print(f"Custom MSE  : {custom_val.item():.6f}")
print(f"nn.MSELoss  : {builtin_val.item():.6f}")
print(f"Match       : {torch.isclose(custom_val, builtin_val).item()}")

### Lab 4 — Implement custom loss functions

**Part A** — plain function:  
Implement `huber_loss(preds, targets, delta=1.0)`.  
Huber loss is MSE for small errors and MAE for large ones — robust to outliers:

```
L(e) = 0.5 * e^2           if |e| <= delta
       delta * (|e| - 0.5 * delta)   otherwise
```

**Part B** — `nn.Module` subclass:  
Implement `RegularizedMSELoss(lambda_l2)` whose `forward(preds, targets, model)` returns:
```
MSE(preds, targets)  +  lambda_l2 * sum(p^2 for all model params)
```

In [ ]:
# Lab 4A: Huber loss as a plain function

def huber_loss(preds, targets, delta=1.0):
    """Element-wise Huber loss, averaged over the batch.

    Args:
        preds:   predicted tensor, any shape
        targets: ground-truth tensor, same shape as preds
        delta:   threshold between MSE and MAE regions

    Returns:
        scalar tensor
    """
    error = None  # YOUR CODE — compute element-wise absolute error
    loss  = None  # YOUR CODE — apply the Huber formula, then .mean()
    return loss


# Lab 4B: regularized MSE as an nn.Module

class RegularizedMSELoss(nn.Module):
    """MSE + L2 weight regularization."""

    def __init__(self, lambda_l2=1e-4):
        super().__init__()
        # Store the regularization strength
        self.lambda_l2 = None  # YOUR CODE

    def forward(self, preds, targets, model):
        # 1. Compute plain MSE
        mse = None  # YOUR CODE

        # 2. Compute sum of squared parameters across all model parameters
        #    Hint: iterate model.parameters(), sum p.pow(2).sum() for each
        l2_penalty = None  # YOUR CODE

        # 3. Return mse + lambda_l2 * l2_penalty
        return None  # YOUR CODE


# --- Verification ---
p = torch.tensor([0.0, 2.0, -2.0])
t = torch.tensor([0.0, 0.0,  0.0])

if huber_loss(p, t) is not None:
    h = huber_loss(p, t, delta=1.0)
    print(f"Huber loss (delta=1): {h.item():.6f}   (expected ~1.1667)")
    # Compare with PyTorch's HuberLoss to verify
    ref = nn.HuberLoss(delta=1.0)(p, t)
    print(f"nn.HuberLoss ref   : {ref.item():.6f}")
    print(f"Match              : {torch.isclose(h, ref).item()}")

tiny_model = nn.Linear(2, 1)
if RegularizedMSELoss.__init__ is not nn.Module.__init__:
    reg_loss_fn = RegularizedMSELoss(lambda_l2=0.0)   # lambda=0 should equal plain MSE
    p2 = torch.tensor([[1.0], [2.0]])
    t2 = torch.tensor([[1.0], [2.0]])
    reg_val = reg_loss_fn(p2, t2, tiny_model)
    print(f"\nRegularizedMSE (lambda=0, perfect preds): {reg_val.item():.6f}   (expected 0.0)")

## Section 5 — Compiling and Training (the PyTorch Training Loop)

### The 6 lines that rule everything

In Keras you call `model.compile(...)` and `model.fit(...)`. PyTorch has no equivalent of `fit` — you write the loop yourself. That is a feature, not a bug: it makes every part of training explicit and debuggable.

The canonical loop is exactly 6 lines per batch:

```python
for x, y in dataloader:
    x, y = x.to(device), y.to(device)  # 1. move data to GPU
    optimizer.zero_grad()               # 2. clear stale gradients
    predictions = model(x)             # 3. forward pass
    loss = loss_fn(predictions, y)     # 4. compute loss
    loss.backward()                    # 5. backpropagate
    optimizer.step()                   # 6. update weights
```

**The #1 bug beginners write:** forgetting `optimizer.zero_grad()`. PyTorch accumulates gradients by default (useful for gradient checkpointing and RNNs). Skip `zero_grad` and your update is the sum of the current batch's gradient *plus every previous batch's gradient*. Training slowly diverges and you wonder why your loss explodes. Make `zero_grad` muscle memory.

### Optimizer choice

| Optimizer | When to use |
|---|---|
| `SGD(lr=0.01)` | Baseline; fine-tunes well with a scheduler |
| `Adam(lr=1e-3)` | Default for most tasks; adaptive per-parameter LR |
| `AdamW(lr=1e-3, weight_decay=0.01)` | Adam + proper L2 regularization; default for transformers |

We will use Adam here.

### Demo — train the sine-wave regressor for 20 epochs

In [ ]:
# Demo: train a tiny MLP to fit sin(x)

# ---- Hyperparameters ----
SINE_EPOCHS = 20
SINE_LR     = 1e-3
SINE_BATCH  = 64

# ---- Model: 1 input -> 64 hidden -> 64 hidden -> 1 output ----
sine_model = nn.Sequential(
    nn.Linear(1, 64),
    nn.ReLU(),
    nn.Linear(64, 64),
    nn.ReLU(),
    nn.Linear(64, 1),
).to(device)

# ---- DataLoader ----
sine_dataset = TensorDataset(x_tensor, y_tensor)
sine_loader  = DataLoader(sine_dataset, batch_size=SINE_BATCH, shuffle=True)

# ---- Loss and optimizer ----
sine_loss_fn  = nn.MSELoss()
sine_optimizer = torch.optim.Adam(sine_model.parameters(), lr=SINE_LR)

# ---- Training loop ----
train_losses = []
for epoch in range(SINE_EPOCHS):
    sine_model.train()          # enable dropout/batchnorm (not needed here, good habit)
    epoch_loss = 0.0
    for xb, yb in sine_loader:
        xb, yb = xb.to(device), yb.to(device)  # 1. device
        sine_optimizer.zero_grad()              # 2. clear grads
        preds = sine_model(xb)                  # 3. forward
        loss  = sine_loss_fn(preds, yb)         # 4. loss
        loss.backward()                         # 5. backward
        sine_optimizer.step()                   # 6. update
        epoch_loss += loss.item()
    avg = epoch_loss / len(sine_loader)
    train_losses.append(avg)
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:2d}/{SINE_EPOCHS} — MSE: {avg:.6f}")

# ---- Plot fit ----
sine_model.eval()
with torch.no_grad():
    x_plot = torch.from_numpy(x_np).unsqueeze(1).to(device)
    y_pred = sine_model(x_plot).cpu().numpy().squeeze()

plt.figure(figsize=(9, 3))
plt.scatter(x_np[::5], y_np[::5], s=5, alpha=0.3, label='noisy data')
plt.plot(x_np, np.sin(x_np), 'r-', lw=2, label='true sin(x)')
plt.plot(x_np, y_pred, 'g-', lw=2, label='model fit')
plt.legend(); plt.title('Sine regression — model fit'); plt.tight_layout(); plt.show()

### Lab 5 — Write the MNIST training loop

Use the `mnist_sequential` model you built in Lab 2.

**Tasks:**
1. Create a `DataLoader` for `mnist_train` with `batch_size=256` and `shuffle=True`.
2. Create a `DataLoader` for `mnist_test` with `batch_size=256` and `shuffle=False`.
3. Define `optimizer = torch.optim.Adam(...)` and `loss_fn = nn.CrossEntropyLoss()`.
4. Write the training loop for **3 epochs**. After each epoch, compute and print the test accuracy.

Test accuracy after 3 epochs should be above **97%**.

**Accuracy formula:**
```python
correct = (predictions.argmax(dim=1) == labels).sum().item()
accuracy = correct / total
```

In [ ]:
# Lab 5: MNIST training loop

# Hyperparameters
MNIST_EPOCHS = 3
MNIST_LR     = 1e-3
MNIST_BATCH  = 256

# 1. DataLoaders
train_loader = None  # YOUR CODE — DataLoader(mnist_train, batch_size=MNIST_BATCH, shuffle=True)
test_loader  = None  # YOUR CODE — DataLoader(mnist_test,  batch_size=MNIST_BATCH, shuffle=False)

# 2. Move model to device (reuse mnist_sequential from Lab 2)
if mnist_sequential is not None:
    mnist_model = mnist_sequential.to(device)

    # 3. Loss function and optimizer
    loss_fn   = None  # YOUR CODE — nn.CrossEntropyLoss()
    optimizer = None  # YOUR CODE — torch.optim.Adam(mnist_model.parameters(), lr=MNIST_LR)

    # 4. Training loop
    for epoch in range(MNIST_EPOCHS):
        mnist_model.train()
        epoch_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            # Step 2: clear gradients
            None  # YOUR CODE

            # Step 3: forward pass
            logits = None  # YOUR CODE

            # Step 4: compute loss
            loss = None  # YOUR CODE

            # Step 5: backprop
            None  # YOUR CODE

            # Step 6: update
            None  # YOUR CODE

            epoch_loss += loss.item()

        # ---- Evaluation (no gradient needed) ----
        mnist_model.eval()
        correct = 0
        total   = 0
        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                logits = None  # YOUR CODE — forward pass
                preds  = None  # YOUR CODE — argmax over class dimension (dim=1)
                correct += (preds == labels).sum().item()
                total   += labels.size(0)

        avg_loss = epoch_loss / len(train_loader)
        accuracy = correct / total
        print(f"Epoch {epoch+1}/{MNIST_EPOCHS} — loss: {avg_loss:.4f} | test acc: {accuracy:.4f}")

## Section 6 — Freezing Layers

### Why freeze?

When you load a pretrained model (BERT, ResNet, GPT-2…) you don't want to destroy the representations that took days of compute to learn. The typical workflow:

1. **Load pretrained weights.**
2. **Freeze the backbone** — set `requires_grad = False` on every parameter. The optimizer ignores frozen parameters.
3. **Add a fresh head** (a new `nn.Linear` for your specific task).
4. **Train only the head** for a few epochs (fast, low-risk).
5. **Optionally unfreeze** some backbone layers and fine-tune at a very low learning rate.

This is called **transfer learning** and it is how every production NLP model is deployed.

### The mechanics

```python
# Freeze all parameters of a module
for param in model.parameters():
    param.requires_grad = False

# Freeze just one layer
for param in model.some_layer.parameters():
    param.requires_grad = False

# Count only the trainable parameters
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
```

When you create a new `optimizer`, pass only the trainable parameters:

```python
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3
)
```

### Demo — freeze the backbone, train only the head

In [ ]:
# Demo: simulating "pretrained backbone + new head" with freezing

class TransferModel(nn.Module):
    """Backbone (frozen pretrained features) + fresh classifier head."""

    def __init__(self):
        super().__init__()
        # Imagine this is a pretrained encoder (we just init randomly for the demo)
        self.backbone = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
        )
        # Fresh head — this is the only part we want to train
        self.head = nn.Linear(128, 10)

    def forward(self, x):
        features = self.backbone(x)   # extract features (frozen)
        return self.head(features)     # classify (trainable)

transfer_model = TransferModel().to(device)

# Count before freezing
total_before = sum(p.numel() for p in transfer_model.parameters())
print(f"Total parameters        : {total_before:,}")

# ---- Freeze the backbone ----
for param in transfer_model.backbone.parameters():
    param.requires_grad = False     # gradient will NOT be computed for these

# Count after freezing
trainable_after = sum(p.numel() for p in transfer_model.parameters() if p.requires_grad)
frozen_after    = sum(p.numel() for p in transfer_model.parameters() if not p.requires_grad)
print(f"Trainable after freeze  : {trainable_after:,}   (head only: 128*10 + 10 = 1,290)")
print(f"Frozen after freeze     : {frozen_after:,}")

# ---- Optimizer only sees trainable parameters ----
transfer_optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, transfer_model.parameters()),
    lr=1e-3,
)
print(f"\nOptimizer param groups  : {len(transfer_optimizer.param_groups)}")
print(f"Params in optimizer     : {sum(p.numel() for g in transfer_optimizer.param_groups for p in g['params']):,}")

### Lab 6 — Freeze, verify, and unfreeze

Given the `FreezeModel` class below (already defined), your tasks are:

1. **Freeze** `model.encoder` completely.
2. **Verify**: print the number of trainable parameters (should match only the `decoder` head).
3. **Build an optimizer** that only touches trainable parameters.
4. **Unfreeze** all parameters with a single loop.
5. **Re-verify**: confirm the full parameter count is trainable again.

In [ ]:
# Lab 6: freezing and unfreezing layers

class FreezeModel(nn.Module):
    """Simple model: encoder (pretend it is pretrained) + decoder head."""
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 256),
            nn.ReLU(),
            nn.Linear(256, 64),
            nn.ReLU(),
        )
        self.decoder = nn.Linear(64, 10)  # fresh head

    def forward(self, x):
        return self.decoder(self.encoder(x))

model = FreezeModel()
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters (all trainable at start): {total_params:,}")

# --- Task 1: Freeze model.encoder ---
# YOUR CODE: loop over model.encoder.parameters() and set requires_grad = False
None  # YOUR CODE

# --- Task 2: Verify trainable count ---
trainable = None  # YOUR CODE — sum of numel for params where requires_grad is True
print(f"Trainable after freezing encoder: {trainable}   (expected 650)")

# --- Task 3: Build optimizer that only covers trainable params ---
lab6_optimizer = None  # YOUR CODE — torch.optim.Adam(filter(...), lr=1e-3)
if lab6_optimizer is not None:
    n_opt = sum(p.numel() for g in lab6_optimizer.param_groups for p in g['params'])
    print(f"Parameters in optimizer: {n_opt}   (expected 650)")

# --- Task 4: Unfreeze all parameters ---
# YOUR CODE: loop over ALL model.parameters() and set requires_grad = True
None  # YOUR CODE

# --- Task 5: Re-verify full trainable count ---
trainable_after_unfreeze = None  # YOUR CODE
print(f"Trainable after unfreezing: {trainable_after_unfreeze}   (expected {total_params})")

## Optional Lab — End-to-End: Train the Transfer Model

You have all the pieces now. Chain everything together:

1. Instantiate a fresh `TransferModel` (or your `FreezeModel`).
2. Freeze the backbone.
3. Create an optimizer over trainable params only.
4. Train for 2 epochs on MNIST (`train_loader`).
5. Evaluate on `test_loader`.
6. Unfreeze everything and fine-tune for 1 more epoch at LR=1e-4.
7. Compare accuracy before and after fine-tuning.

Expected result: freezing gives ~90–92%; fine-tuning after unfreezing climbs to ~97%+.

In [ ]:
# Optional Lab: end-to-end transfer learning on MNIST

# YOUR CODE below — use the building blocks from Labs 2, 5, and 6

# Step 1: fresh model + freeze backbone
opt_model = None  # YOUR CODE

# Step 2: freeze backbone, build optimizer
# YOUR CODE

# Step 3: train for 2 epochs (head only)
# YOUR CODE

# Step 4: evaluate
# YOUR CODE

# Step 5: unfreeze and fine-tune for 1 epoch at lr=1e-4
# YOUR CODE

# Step 6: evaluate again and print comparison
# YOUR CODE
print("Optional lab complete!")

## Congratulations!

You have now built every primitive that the rest of this course relies on.

### What you practiced

| Section | PyTorch concept |
|---|---|
| 1 | `nn.Linear`, `nn.ReLU`, `nn.Dropout`, `nn.Conv2d` |
| 2 | `nn.Sequential` — linear pipelines with shape arithmetic |
| 3 | `nn.Module` subclassing — skip connections, multi-branch models |
| 4 | Custom loss functions — Huber loss, regularized MSE |
| 5 | The 6-line training loop — `zero_grad`, forward, loss, backward, step |
| 6 | Freezing — `requires_grad = False`, transfer learning pattern |

### Key takeaways

1. **`optimizer.zero_grad()` must be first in the loop.** Forgetting it is the most common PyTorch bug.
2. **Never apply softmax before `CrossEntropyLoss`.** It already applies log-softmax internally.
3. **`model.train()` vs `model.eval()`**: switches Dropout and BatchNorm between training and inference modes. Always set correctly.
4. **Frozen parameters are invisible to the optimizer.** Use `filter(lambda p: p.requires_grad, model.parameters())` when creating the optimizer.
5. **`torch.no_grad()` during evaluation** — disables the autograd engine and cuts memory usage in half.

### Self-check questions

1. What happens if you pass `nn.Sigmoid()(logits)` to `nn.CrossEntropyLoss()`? Why is it wrong?
2. Why does `nn.Sequential` fail when you need a skip connection?
3. You have a model with 10M parameters. You freeze 9.9M of them and train only a 100K head. How much faster would you expect training to be?

### Next up

**Notebook 5 — CBOW Word Embeddings**: you will use `nn.Embedding` (a new layer type), build a `Dataset` and `DataLoader` from scratch, and apply this exact training loop to learn dense word vectors from Yelp reviews.